In [1]:
RERUN = False
from _common import *
from _dna_helpers import *

P = paper()
J = "dna/june_split65k_recovered"          # the recovered June-2026 Table 6 records
G = RESULTS / "dna" / "regen"              # the regenerated Evo 2 results
print("Evo 2 results:", G / "table6", "|",
      "present" if (G / "table6" / "evo2_summary.csv").exists() else "pending (job 49298971)")

Evo 2 results: /n/netscratch/albergo_lab/Lab/uunneberg/MultiMFM/results/dna/regen/table6 | pending (job 49298971)


# 09 · Evo 2 sequence-model plausibility (Table 6)

Paper App. B.1, Table 6. This notebook is Table 6 only, per the plan's §5 table; the regenerated Fig 6, Table 16,
Fig 7 and Fig 8 material lives in notebooks 02 and 03.

Two things are shown side by side:

1. **The submitted numbers**, checked against the only surviving record of them — the run directory and tracked
   summary CSV were purged, but the complete summary CSV survives verbatim in a Codex transcript and is stored at
   `results/dna/june_split65k_recovered/`. Per-sequence scores were never printed, so the published CIs cannot be
   recomputed; everything else is verified against that CSV.
2. **A regeneration** with the same scorer and the same Evo 2 model on the sample sets that still exist. The
   published inputs — the `split65k` sample sets, split file and models — are all purged, so the published numbers
   cannot be reproduced exactly at all; what is regenerated is the protocol applied to the parent-disjoint
   n = 1,000 C0 sets, the same sets that Tables 4 and 5 characterise.

`RERUN = False` rebuilds everything below from `results/` on CPU. The Evo 2 scoring itself runs on one GPU in a
separate environment (`scripts/dna/regen/evo2_environment.yml` + `evo2_postinstall.py`) and never in this kernel.

## Table 6 — Evo 2 sequence-model plausibility

Paper App. B.1, Table 6: Evo 2 negative log-likelihood of the C0-steered L = 50 sets (targets ±1, guided and
unguided, n = 1,000 each) and of four 1,000-window real-data samples. Per sequence,
NLL = −½ (mean log p(xᵢ | x_<ᵢ) over i = 2…L on the sequence + the same on its reverse complement), so 49 scored
bases; 95 % CI = percentile bootstrap over sequences (2,000 resamples); perplexity = exp(mean NLL);
Δ = NLL − NLL(`split_test_sample`).

**Where the published numbers come from.** The run directory (`workdir/yeast_split_dmfm_c0_evo2_2026-06-10/`) and
the tracked summary CSV of the DNA repo were purged. The complete summary CSV survives verbatim in a Codex
transcript and is stored as `results/dna/june_split65k_recovered/yeast_split_dmfm_c0_evo2_summary_2026-06-10.csv`.
Per-sequence Evo 2 scores were not printed anywhere, so the published CIs cannot be recomputed; everything else in
Table 6 is checked against that CSV below.

**Which model and sets (discrepancy, for the authors).** App. B.1 says the samples come from the parent-disjoint
L = 50 dMFM and that novelty is measured against the 58 held-out test parents. The recovered records say otherwise:
the scored sets are the June 2026 runs `workdir/yeast_split_dmfm_c0_samples_target_{p1,m1}_2026-06-02` with the
**`split65k`** teacher/dMFM, guided by `dinko/C0free_torch.pt` (not the parent-disjoint guide), mc = 16, and the
reference samples are drawn from that random 65,000 / 8,000 / 9,368 window split of `data/yeast_mid50.pt`. Same for
Tables 4–5 (notebook 03).

In [2]:
ev = load_csv(f"{J}/yeast_split_dmfm_c0_evo2_summary_2026-06-10.csv")
ev

,source,set,target_c0,n_samples,seq_len,scored_positions,orientation,evo2_mean_logprob_mean,evo2_nll_mean,evo2_nll_std,evo2_nll_median,evo2_nll_mean_ci95_low,evo2_nll_mean_ci95_high,evo2_perplexity_mean,evo2_perplexity_from_mean_nll,evo2_perplexity_median,evo2_nll_delta_vs_split_test
0,generated,guided,-1.0,1000,50,49.0,forward_rc_mean,-1.235049,1.235049,0.153504,1.282366,1.225210,1.244503,3.475296,3.438547,3.605160,-0.110354
1,generated,guided,1.0,1000,50,49.0,forward_rc_mean,-1.352912,1.352912,0.052102,1.358638,1.349743,1.356084,3.873791,3.868673,3.890889,0.007509
2,generated,unguided,-1.0,1000,50,49.0,forward_rc_mean,-1.361305,1.361305,0.047016,1.365952,1.358407,1.364136,3.905505,3.901282,3.919452,0.015902
3,generated,unguided,1.0,1000,50,49.0,forward_rc_mean,-1.361939,1.361939,0.045786,1.364856,1.359064,1.364609,3.907804,3.903757,3.915158,0.016537
4,reference,full_sample,NaN,1000,50,49.0,forward_rc_mean,-1.347692,1.347692,0.066560,1.358877,1.343466,1.351716,3.856726,3.848531,3.891819,0.002289
5,reference,split_test_sample,NaN,1000,50,49.0,forward_rc_mean,-1.345403,1.345403,0.077512,1.359255,1.340583,1.350102,3.850566,3.839734,3.893293,0.000000
6,reference,split_train_sample,NaN,1000,50,49.0,forward_rc_mean,-1.346349,1.346349,0.069729,1.358657,1.342004,1.350746,3.852240,3.843370,3.890966,0.000947
7,reference,split_val_sample,NaN,1000,50,49.0,forward_rc_mean,-1.338897,1.338897,0.085114,1.354572,1.333741,1.344110,3.827815,3.814833,3.875102,-0.006506


## Table 6 — paper vs recovered summary CSV
Row mapping: paper `generated_guided_±1` ↔ (`generated`, `guided`, target ±1), etc. Every printed cell is compared
at the paper's printed precision (4 decimals; `n` and bases exactly).

In [3]:
t6 = P["table6_evo2_nll"]
cols = t6["_columns"]
src_cols = {"n": "n_samples", "bases": "scored_positions", "nll_mean": "evo2_nll_mean", "ci_low": "evo2_nll_mean_ci95_low",
            "ci_high": "evo2_nll_mean_ci95_high", "perplexity": "evo2_perplexity_from_mean_nll",
            "delta_heldout": "evo2_nll_delta_vs_split_test"}


def ev_row(key: str) -> pd.Series:
    if key.startswith("generated_"):
        _, setname, tgt = key.split("_")
        m = (ev.source == "generated") & (ev.set == setname) & (ev.target_c0 == float(tgt))
    else:
        m = (ev.source == "reference") & (ev.set == key[len("reference_"):])
    assert m.sum() == 1, key
    return ev[m].iloc[0]


rows = []
for key, vals in t6.items():
    if key.startswith("_"):
        continue
    r = ev_row(key)
    for c, pv in zip(cols, vals):
        rows.append(dict(item=f"{key} {c}", paper=pv, ours=float(r[src_cols[c]]), atol=0 if c in ("n", "bases") else None,
                         dec=None if c in ("n", "bases") else 4))
t6cmp = show(cmp(rows))
t6cmp

status counts: {'match': 56}


,item,paper,ours,status,tol,tol_src,note
0,generated_guided_-1 n,1000.0000,1000.000000,match,0.00000,atol,
1,generated_guided_-1 bases,49.0000,49.000000,match,0.00000,atol,
2,generated_guided_-1 nll_mean,1.2350,1.235049,match,0.00005,atol,
3,generated_guided_-1 ci_low,1.2252,1.225210,match,0.00005,atol,
4,generated_guided_-1 ci_high,1.2445,1.244503,match,0.00005,atol,
5,generated_guided_-1 perplexity,3.4385,3.438547,match,0.00005,atol,
6,generated_guided_-1 delta_heldout,-0.1104,-0.110354,match,0.00005,atol,
7,generated_unguided_-1 n,1000.0000,1000.000000,match,0.00000,atol,
8,generated_unguided_-1 bases,49.0000,49.000000,match,0.00000,atol,
9,generated_unguided_-1 nll_mean,1.3613,1.361305,match,0.00005,atol,


## Text of App. B.1
"the guided samples have mean NLL 1.3529, close to the held-out split_test_sample value of 1.3454 with Δ = 0.0075 …
the bootstrap intervals overlap. For target_c0 = −1 … 1.2350 versus 1.3454 with Δ = −0.1104".

In [4]:
g_p, g_m, held = ev_row("generated_guided_1.0"), ev_row("generated_guided_-1.0"), ev_row("reference_split_test_sample")
tx = P["appB1_text"]
rows = [
    dict(item="+1 guided NLL", paper=tx["plus1_guided_nll"], ours=float(g_p.evo2_nll_mean), dec=4),
    dict(item="held-out split_test_sample NLL", paper=tx["heldout_nll"], ours=float(held.evo2_nll_mean), dec=4),
    dict(item="+1 guided delta", paper=tx["plus1_guided_delta"], ours=float(g_p.evo2_nll_delta_vs_split_test), dec=4),
    dict(item="-1 guided NLL", paper=tx["minus1_guided_nll"], ours=float(g_m.evo2_nll_mean), dec=4),
    dict(item="-1 guided delta", paper=tx["minus1_guided_delta"], ours=float(g_m.evo2_nll_delta_vs_split_test), dec=4),
    dict(item="+1 guided CI overlaps held-out CI (1 = yes)", paper=1,
         ours=float(g_p.evo2_nll_mean_ci95_low <= held.evo2_nll_mean_ci95_high and held.evo2_nll_mean_ci95_low <= g_p.evo2_nll_mean_ci95_high), atol=0),
]
show(cmp(rows))

status counts: {'match': 6}


,item,paper,ours,status,tol,tol_src,note
0,+1 guided NLL,1.3529,1.352912,match,0.00005,atol,
1,held-out split_test_sample NLL,1.3454,1.345403,match,0.00005,atol,
2,+1 guided delta,0.0075,0.007509,match,0.00005,atol,
3,-1 guided NLL,1.2350,1.235049,match,0.00005,atol,
4,-1 guided delta,-0.1104,-0.110354,match,0.00005,atol,
5,+1 guided CI overlaps held-out CI (1 = yes),1.0000,1.000000,match,0.00000,atol,


## Internal consistency of the recovered CSV (full precision)
perplexity_from_mean_nll = exp(mean NLL); Δ = NLL − NLL(split_test_sample); mean_logprob = −NLL. These hold to
floating-point precision, which is some evidence the transcript copy is intact (it also matches the paper to all
printed digits above). The "n = 1000" of every row and "49 scored bases" (= L − 1) are as printed.

In [5]:
held_nll = float(held.evo2_nll_mean)
chk = []
for _, r in ev.iterrows():
    name = f"{r.source}/{r.set}/{r.target_c0}"
    chk += [dict(item=f"{name} exp(NLL)", paper=float(r.evo2_perplexity_from_mean_nll), ours=float(np.exp(r.evo2_nll_mean)), atol=1e-9),
            dict(item=f"{name} delta", paper=float(r.evo2_nll_delta_vs_split_test), ours=float(r.evo2_nll_mean - held_nll), atol=1e-12),
            dict(item=f"{name} -mean_logprob", paper=float(r.evo2_nll_mean), ours=float(-r.evo2_mean_logprob_mean), atol=1e-12)]
show(cmp(chk)).rename(columns={"paper": "stored"})

status counts: {'match': 24}


,item,stored,ours,status,tol,tol_src,note
0,generated/guided/-1.0 exp(NLL),3.438547,3.438547,match,1.000000e-09,atol,
1,generated/guided/-1.0 delta,-0.110354,-0.110354,match,1.000000e-12,atol,
2,generated/guided/-1.0 -mean_logprob,1.235049,1.235049,match,1.000000e-12,atol,
3,generated/guided/1.0 exp(NLL),3.868673,3.868673,match,1.000000e-09,atol,
4,generated/guided/1.0 delta,0.007509,0.007509,match,1.000000e-12,atol,
5,generated/guided/1.0 -mean_logprob,1.352912,1.352912,match,1.000000e-12,atol,
6,generated/unguided/-1.0 exp(NLL),3.901282,3.901282,match,1.000000e-09,atol,
7,generated/unguided/-1.0 delta,0.015902,0.015902,match,1.000000e-12,atol,
8,generated/unguided/-1.0 -mean_logprob,1.361305,1.361305,match,1.000000e-12,atol,
9,generated/unguided/1.0 exp(NLL),3.903757,3.903757,match,1.000000e-09,atol,


## The scored sets (recovered sampling records)
The C0 scores of the same June sets (guide regressor `C0free_torch.pt`, from each run's `summary.json`) and their
sampling arguments, both recovered from the same transcript. The unguided sets differ between the two targets
(different seeds, 0 and 1), which is why Table 5 has two different unguided rows.

In [6]:
summ = []
for tag in ["p1", "m1"]:
    a = json.loads((RESULTS / J / f"samples_target_{tag}_2026-06-02.args.json").read_text())
    s = json.loads((RESULTS / J / f"samples_target_{tag}_2026-06-02.summary.json").read_text())
    summ.append({"target_c0": a["target_c0"], "seed": a["seed"], "n": a["n_samples"], "mc": a["mc"], "guidance_frac": a["guidance_frac"],
                 "guide_t": f"[{a['guide_t_start']}, {a['guide_t_end']}]", "reward_sigma": a["reward_sigma"], "reward_scale": a["reward_scale"],
                 "guide": a["c0_ckpt"], "teacher": a["teacher_ckpt"].split("/")[1], "student": "/".join(a["student_ckpt"].split("/")[1:]),
                 "guide MAE unguided": s["unguided"]["mae"], "guide MAE guided": s["guided"]["mae"], "frac improved": s["delta"]["frac_improved"]})
pd.DataFrame(summ).T

,0,1
target_c0,1.0,-1.0
seed,0,1
n,1000,1000
mc,16,16
guidance_frac,8.0,8.0
guide_t,"[0.01, 0.95]","[0.01, 0.95]"
reward_sigma,0.15,0.15
reward_scale,1.0,1.0
guide,dinko/C0free_torch.pt,dinko/C0free_torch.pt
teacher,yeast_mid50_dfm_dit_split65k_h192_b4_lr5e-5_2026-05-12_16-47-35,yeast_mid50_dfm_dit_split65k_h192_b4_lr5e-5_2026-05-12_16-47-35


## Related Evo 2 numbers on the parent-disjoint models (not Table 6)
The exact-reward experiment (notebook 04, Table 13) scored parent-disjoint samples with the same method
(`score_parent_reward_attainment_evo2.py`, which imports the NLL function of the C0-set scorer). For orientation, the
L = 50 values (100 sequences per set; different model, sets and n from Table 6):

In [7]:
out = []
for reward in ["motif", "conjunction"]:
    s = load_csv(f"dna/exact_rewards/{reward}/L50/evo2/evo2_summary.csv")
    out.append(s.assign(experiment=f"exact reward: {reward}"))
pd.concat(out)[["experiment", "set", "n_sequences", "scored_positions", "orientation", "evo2_nll_mean", "evo2_nll_ci95_low",
                "evo2_nll_ci95_high", "evo2_perplexity_from_mean_nll", "evo2_nll_delta_vs_heldout_test"]]

,experiment,set,n_sequences,scored_positions,orientation,evo2_nll_mean,evo2_nll_ci95_low,evo2_nll_ci95_high,evo2_perplexity_from_mean_nll,evo2_nll_delta_vs_heldout_test
0,exact reward: motif,unguided,100,49,forward_rc_mean,1.359098,1.348807,1.368860,3.892679,0.034361
1,exact reward: motif,guided,100,49,forward_rc_mean,1.343611,1.333465,1.352483,3.832859,0.018874
2,exact reward: motif,heldout_test,100,49,forward_rc_mean,1.324737,1.303495,1.343983,3.761195,0.000000
0,exact reward: conjunction,unguided,100,49,forward_rc_mean,1.359098,1.348807,1.368860,3.892679,0.034361
1,exact reward: conjunction,guided,100,49,forward_rc_mean,1.337228,1.329466,1.344779,3.808472,0.012492
2,exact reward: conjunction,heldout_test,100,49,forward_rc_mean,1.324737,1.303495,1.343983,3.761195,0.000000


### Regenerated Table 6 *(rerun of the recovered scorer on the parent-disjoint sets)*

**The scorer is the original, not a reconstruction.** In `dirichlet-flow-matching` only
`scripts/__pycache__/score_yeast_c0_evo2.cpython-39.pyc` survived, so `dmfm.experiments.score_yeast_c0_evo2` was
first rebuilt from the Codex session that wrote it. The upstream source has since been read from the durable clone
of `tullebulle/DNA-MFM@187fe7b` on holylabs, and the module is now **derived from that file** by a scripted list of
anchored edits (verbatim copy + sha256 in `results/dna/regen/original/`). Re-deriving changed no executable
statement — the two are identical under `ast.dump` with docstrings stripped — so nothing below needed rerunning.

Of the 18 functions, **13 are AST-identical to the original**: `prepare_batch`, `score_mean_logprobs`, `score_rows`,
`bootstrap_ci`, `summarize`, `reverse_complement`, `tokens_to_seqs`, `model_forward`, `load_evo2`, `generated_rows`,
`cap_df`, `run_metadata`, `load_args_payload`. That is the whole scoring path: tokenisation, padding, the optional
BOS, windowing, strand handling, the reduction, what is averaged, the bootstrap and every summary column. The five
that differ (`parse_args`, `load_train_tokens`, `discover_csvs`, `baseline_rows`, `main`) differ only in imports,
repo-relative defaults and CLI wiring; `baseline_rows`' new `split_names` argument still *defaults* to the
original `('train', 'val', 'test')` keys. Every deviation is enumerated in the module docstring.

**The inputs are not.** The June `split65k` sample sets and the split file
`data/yeast_mid50_split_seed0_train65000_val8000.pt` are purged, and so are the models that made them, so the
published sets cannot be rescored *at all*. What is regenerated here is the same scorer, the same Evo 2 model and
the same protocol applied to the sets that do exist: the parent-disjoint n = 1,000 C0 runs of Table 1
(`results/dna/c0_guidance/n1000/target_{1p0,m1p0}/sample_scores.csv`) and four 1,000-window real-data samples drawn
with seed 0 from `data/dna/yeast_parent_disjoint/` (`a_train` / `a_val` / `test` / full).

**Environment and weights, both rebuilt.** `scripts/dna/regen/evo2_environment.yml` (Python 3.12, torch 2.7.1+cu128,
`evo2` 0.5.5, `vtx` 1.0.8) and `arcinstitute/evo2_7b_base` revision `074097e9dc788e8bfe045d6495b9f6153a7c6bfc`
(13 GB, the exact revision the paper runs used), cached under `cache/xdg/huggingface`. Scoring runs on one H100 in
that separate environment — not in the `multimfm` kernel:
`MODE=c0 OUT_DIR=outputs/dna/regen/table6 sbatch scripts/dna/regen/score_evo2.sbatch`.

In [8]:
t6_new = G / "table6" / "evo2_summary.csv"
if t6_new.exists():
    ev_new = pd.read_csv(t6_new)
    display(ev_new)
else:
    ev_new = None
    print("pending: results/dna/regen/table6/evo2_summary.csv "
          "(MODE=c0 OUT_DIR=outputs/dna/regen/table6 sbatch scripts/dna/regen/score_evo2.sbatch)")

pending: results/dna/regen/table6/evo2_summary.csv (MODE=c0 OUT_DIR=outputs/dna/regen/table6 sbatch scripts/dna/regen/score_evo2.sbatch)


In [9]:
if ev_new is not None:
    t6 = P["table6_evo2_nll"]
    rows = []
    for key, (src, setname, tgt) in {
        "generated_guided_-1.0": ("generated", "guided", -1.0),
        "generated_unguided_-1.0": ("generated", "unguided", -1.0),
        "generated_guided_1.0": ("generated", "guided", 1.0),
        "generated_unguided_1.0": ("generated", "unguided", 1.0),
        "reference_split_test_sample": ("reference", "split_test_sample", None),
        "reference_split_train_sample": ("reference", "split_train_sample", None),
        "reference_split_val_sample": ("reference", "split_val_sample", None),
        "reference_full_sample": ("reference", "full_sample", None),
    }.items():
        m = (ev_new.source == src) & (ev_new.set == setname)
        if tgt is not None:
            m &= (ev_new.target_c0 == tgt)
        if not m.any():
            continue
        r = ev_new[m].iloc[0]
        paper_row = dict(zip(t6["_columns"], t6[key]))
        rows.append(dict(item=f"{key} · NLL", paper=paper_row["nll_mean"], ours=float(r.evo2_nll_mean), dec=4))
        rows.append(dict(item=f"{key} · Δ held-out", paper=paper_row["delta_heldout"],
                         ours=float(r.evo2_nll_delta_vs_split_test), dec=4))
    display(show(cmp(rows)))
else:
    print("pending: regenerated Table 6 comparison")

pending: regenerated Table 6 comparison


**Reading the comparison.** Expect mismatches on every generated row: those sets come from a different generator,
a different guide and a different guidance configuration than the published ones, and the reference rows come from a
different (parent-disjoint) split of the same underlying yeast windows. What the rerun *does* test is the claim of
App. B.1 — whether steered samples stay close to held-out real yeast DNA under an external sequence model, and
whether the target_c0 = −1 direction again scores implausibly *low* relative to held-out data.

### How to reproduce the published Table 6 exactly

Not possible. It needs (a) `workdir/yeast_split_dmfm_c0_samples_target_{p1,m1}_2026-06-02/sample_scores.csv`,
(b) `data/yeast_mid50.pt` + `data/yeast_mid50_split_seed0_train65000_val8000.pt`, and (c) the `split65k` base/dMFM
checkpoints and `dinko/C0free_torch.pt` to resample — all purged, none on Hugging Face, none in the holylabs backup.
The recovered summary CSV above is the only surviving record of those numbers.

In [10]:
root = G / "table6"
print("Evo 2 artifacts under", root.relative_to(RESULTS) if root.exists() else "(pending)")
for f in sorted(root.rglob("*")) if root.exists() else []:
    if f.is_file():
        print(f"  {f.relative_to(root)}  ({f.stat().st_size / 1024:.1f} KiB)")

Evo 2 artifacts under (pending)
